# 第 5 关 · 自回归　★最难的一关

随机顺序解码器。让碱基一个一个定，后定的看得见先定的。

**BOSS：模型自测 8/8（含因果性），且配对有效率 ≥ 第 4 关 + 5 个百分点**

---

**这一关有 7 个部分：**

0. 准备
1. 核心机关：一个乘法
2. 解码层长什么样
3. 因果性自测：先过这一关再训练
4. 训练
5. 生成一条，盯住茎区
6. 第 4 关 vs 第 5 关：自回归买到了什么

## 部分 0 · 准备

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "minif").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("仓库根：", ROOT)

from minif.plotting import setup
plt = setup()

In [ ]:
import numpy as np
import torch

from minif import data, engine, features as F, pairing, paths, scoreboard
from minif.model import MiniMPNN, DecLayer
from minif.plotting import tidy, PALETTE

MOL = "rna"
tr = data.load(MOL, "train"); va = data.load(MOL, "val")
data.describe(tr, "训练集："); data.describe(va, "验证集：")
prev_pv = scoreboard.read()["levels"].get("4", {}).get("pair_validity")
print("第 4 关配对有效率 %.1f%%，今天要到 %.1f%% 以上"
      % (100*prev_pv, 100*(prev_pv+0.05)) if prev_pv else "（第 4 关没记上配对有效率，回去重跑一次）")

## 部分 1 · 核心机关：一个乘法

给核苷酸排一个顺序，一个一个定。定第 i 个的时候，
允许看见「已经定好」的邻居选了什么碱基。整个自回归就这一行：

```python
sj = s_emb[idx] * mask_bw     # 还没轮到的邻居，碱基信息被乘成 0
```

先用一个 6 个核苷酸的小例子，把 `mask_bw` 这个矩阵打出来看清楚。

In [ ]:
L, k = 6, 3
order = torch.tensor([3, 0, 5, 1, 4, 2])     # 解码顺序：先定 3 号，再定 0 号 ...
rank = torch.empty(L, dtype=torch.long); rank[order] = torch.arange(L)
idx_small = torch.tensor([[1,2,3],[0,2,3],[1,3,4],[2,4,5],[3,5,0],[4,0,1]])

print("解码顺序 order =", order.tolist(), " （第 t 个被解码的是谁）")
print("名次     rank  =", rank.tolist(), " （每个核苷酸排第几）")
print()
mask = (rank[idx_small] < rank.unsqueeze(1)).float()
print("mask_bw[i, j] = 1 表示「邻居 j 比 i 先定，i 看得见 j」")
print()
print("      邻居:" + "".join("%8d" % j for j in range(k)))
for i in range(L):
    print("  i=%d(第%d个定)" % (i, rank[i]) + "".join(
        "  %d:%s " % (j, "可见" if m else "挡住") for j, m in zip(idx_small[i].tolist(), mask[i].tolist())))

In [ ]:
print("注意：第一个被解码的 3 号，它的 mask 全是 0——什么都看不见，只能靠结构。")
print("最后被解码的 2 号，能看见所有已定的邻居。")
print()
print("为什么顺序是随机的，不是 5' 到 3'？")
print("  从左到右只能「前面决定后面」。随机顺序下模型见过各种")
print("  「已知一部分、补剩下一部分」的局面——这正是 RNA 设计的真实场景：")
print("  固定几个关键位点，让模型补其余。第 7 关直接吃这个红利。")

## 部分 2 · 解码层长什么样

和编码层几乎一样，区别就是消息里多了「邻居的碱基身份」，
而且这一项对还没轮到的邻居会被乘上 0 抹掉。

In [ ]:
import inspect
print(inspect.getsource(DecLayer))

## 部分 3 · 因果性自测：先过这一关再训练

**这是今天最重要的一格。**

解码器如果偷看到了自己要预测的那个碱基，训练 loss 会好看、恢复率会虚高，
但采样生成时全盘崩掉——而且从训练曲线上完全看不出来。

第 3 项改「排在后面」的位置，看 logits 动不动（必须不动）。
第 4 项比教师强制和逐步解码的结果是否完全一致。

In [ ]:
!cd .. && python tests/test_model.py

In [ ]:
# 自己再手动验一次最关键的那条
model = MiniMPNN(d_node=F.node_dim(MOL), d_edge=F.edge_dim(MOL),
                 n_letters=F.n_letters(MOL)).eval()
V, E, idx, seq = engine.prep(tr[0], MOL)
Ls = seq.shape[0]
order = torch.randperm(Ls)
rank = torch.empty(Ls, dtype=torch.long); rank[order] = torch.arange(Ls)

with torch.no_grad():
    h, e = model.encode(V, E, idx)
    base = model.decode(h, e, idx, seq, order)
    i = int(order[Ls // 2])
    s2 = seq.clone()
    for j in range(Ls):                       # 把排在 i 后面的位置全改掉
        if rank[j] > rank[i]:
            s2[j] = (s2[j] + 1) % 4
    d = (model.decode(h, e, idx, s2, order)[i] - base[i]).abs().max().item()
print("把排在第 i 位之后的碱基全改一遍，logits[i] 变化 %.2e" % d)
print("必须≈0。不为 0 就是解码器漏看了答案，别往下训练。" if d > 1e-6 else "通过。")

## 部分 4 · 训练

教师强制：把真实序列整条喂进去，靠 mask 保证第 i 位看不见自己。
这样不用真的一步步跑（太慢）。

In [ ]:
model = MiniMPNN(d_node=F.node_dim(MOL), d_edge=F.edge_dim(MOL),
                 n_letters=F.n_letters(MOL), d_model=128, d_hidden=128,
                 n_enc=3, n_dec=3)
print("参数量", sum(p.numel() for p in model.parameters()))
best = engine.fit(model, tr, va, MOL, epochs=60, lr=1e-3, noise=0.02,
                  autoregressive=True, log_every=5,
                  save=paths.run_file("level5.pt"), level=5, eval_pairs=True)

## 部分 5 · 生成一条，盯住茎区

低温采样，和天然序列对着看。重点不是逐位像不像，是**括号对上的两个位置配不配**。

In [ ]:
r = max(va, key=lambda x: (x["partner"] >= 0).sum())
des = engine.design(model, r, MOL, temperature=0.1)
dots = "".join("(" if 0 <= p and p > i else (")" if p >= 0 else ".")
               for i, p in enumerate(r["partner"]))
print(r["id"])
print("配对  ", dots)
print("天然  ", r["seq"]); pairing.summarize(r["seq"], r["partner"], "        ")
print("设计  ", des);      pairing.summarize(des, r["partner"], "        ")
print("\n逐位相同 %.0f%%" % (100*sum(a == b for a, b in zip(r["seq"], des))/len(des)))

## 部分 6 · 第 4 关 vs 第 5 关：自回归买到了什么

这张图是整个项目最有说服力的一张。

In [ ]:
lv = scoreboard.read()["levels"]
pv4 = lv.get("4", {}).get("pair_validity"); pv5 = lv.get("5", {}).get("pair_validity")
rc4 = lv.get("4", {}).get("recovery");      rc5 = lv.get("5", {}).get("recovery")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, (a, b, name) in zip(axes, [(rc4, rc5, "序列恢复率"), (pv4, pv5, "配对有效率")]):
    if a is None or b is None:
        ax.set_axis_off(); continue
    vals = [100*a, 100*b]
    ax.bar(["第4关\n独立预测", "第5关\n自回归"], vals, color=PALETTE[:2], width=.55)
    for x, v in enumerate(vals):
        ax.annotate("%.1f%%" % v, (x, v), xytext=(0, 4), textcoords="offset points",
                    ha="center", fontsize=14)
    ax.set_title(name); ax.set_ylabel("%"); ax.set_ylim(0, max(vals)*1.25)
    tidy(ax)
plt.tight_layout(); plt.show()
if pv4 and pv5:
    print("配对有效率涨了 %.1f 个百分点，恢复率只涨了 %.1f 个。"
          % (100*(pv5-pv4), 100*(rc5-rc4)))
    print("自回归买到的主要不是「猜得更准」，是「前后不打架」。")

In [ ]:
!cd .. && python game.py check 5

## 自测三问

1. gRNAde / ProteinMPNN 是怎么生成序列的？要能讲出随机顺序这一点。
2. 教师强制是什么？它和真正采样生成的区别在哪？
3. 如果解码器漏看了答案，你怎么发现？

### 卡住的降级方案

到今天结束还没过 `tests/test_model.py`，**不要顺延**。
`python game.py hint 5` 开参考实现，跑出结果继续往前，第 7 关收口时再回头。
保住主线比补齐每一关重要。

写进 `notes/level5.md`。